# 

Fase 2- Capa de coronas 
## Proceso previo al cambio de lenguaje de programación 

Este notebook crea la capa de puntos con la corona de cada evento. Es la entrada de los dos análisis de la Fase 2 y la ubicación de los eventos en las fases siguientes.

La corona se definió como la celda del DEM de cota máxima cuyo centro cae dentro del polígono (definido en la fase 1). Cada polígono se procesa por separado, leyendo solo la porción del DEM que cubre. Así,  los polígonos que se superponen no se mezclan, cada corona lleva desde el principio el identificador de su evento y, si varias celdas empatan en la cota máxima, se elige una sola.

**Datos de entrada**

|Elemento|Descripción|
|---|---|
|`Inventario_integrado`|Misma capa de poligonos del inventario completo. De aquí salen la geometría, para buscar la corona, y el área. |
|`fase1_movilidad`|Capa de la Fase 1. De aquí salen el grupo de fuente, el tipo de movimiento, la reactivación, H/L y la elongación.|
|`dem_VA`|DEM completo del área de estudio de 2 × 2 m |
|`Slope_VA`|raste de pendiente en grados. Si no se tiene, el código la calcula promediando el DEM a 10 m|

**Salidas**

Genera capas de coronas para su uso y para su revisión en caso de tener otra capa de coronas de comparación. La capa `coronas.shp` es la que se usará para los scripts de R. 


**Cómo reproducir**

1. En el proyecto de ArcGIS Pro cargar las capas de entrada en el mapa, o poner sus rutas completas en la celda de configuración.
2. Dejar en `CARPETA_R` la misma carpeta donde se escribirán los scripts posteriores en R y que usarán los archivos de esa carpeta.
3. Ejecutar las celdas en orden




# configuracion
### Librerías, capas, campos y parámetros

**¿Que hace?**

Carga las librerías y reúne todo lo que se puede modificar: las capas de entrada, los nombres de los campos, los umbrales y dónde se guardan los resultados. Al final fija la geodatabase de salida y la imprime.




In [ ]:
import os
import math
import unicodedata
import numpy as np
import arcpy
from matplotlib.path import Path

# Capas 
CAPA_POLIGONOS = "Inventario_integrado"     # inventario actualizado
CAPA_METRICAS  = "fase1_movilidad"      # capa con todas las métricas calculadas
DEM            = "dem_VA"          # el DEM 

RASTER_PENDIENTE = "Slope_VA" # Pendiente en la corona


# Campos 
CAMPO_ID_POLIGONOS = "ID_evento"         # en CAPA_POLIGONOS
CAMPO_ID_METRICAS  = "ID_evento"         # en CAPA_METRICAS
CAMPO_GRUPO   = "grupo"               # en CAPA_METRICAS: Mapeo / Institucional
CAMPO_TIPO    = "tipo_movim"          # en CAPA_METRICAS
CAMPO_REACTIV = "reactivación"        # en CAPA_METRICAS
CAMPO_HL      = "H_L"                 # en CAPA_METRICAS
CAMPO_ELONG   = "E"                   # en CAPA_METRICAS

UMBRAL_MAPEO = 403.05          # rollover de la Fase 0 (m2)
UMBRAL_INST  = 154.34


# Poligonos que quedan en parte fuera del DEM: se excluyen si les faltamas de esta fraccion de celdas
FRACCION_RECORTE = 0.10

EXCLUIR_MANUAL   = []          # IDs a excluir a mano, si hace falta

MIN_CELDAS       = 10          # se excluyen los poligonos con menos celdas del DEM (Fase 1)

# Salidas
GDB_SALIDA  = None             # None = la geodatabase del proyecto
NOMBRE_CAPA = "coronas_revision"
CARPETA_R   = r"D:\Users\tasal\Desktop\Tesis de grado\fase2"   # aqui va coronas.shp

arcpy.env.overwriteOutput = True
if GDB_SALIDA is None:
    GDB_SALIDA = arcpy.mp.ArcGISProject("CURRENT").defaultGeodatabase
print("Geodatabase de salida:", GDB_SALIDA)


# Funciones 

**¿Que hace?**

Define las funciones que usan las celdas siguientes: dos para encontrar capas y campos, y tres para encontrar la corona dentro de cada polígono.

* `ruta(capa)` devuelve la ruta completa de una capa a partir de su nombre en el mapa
* `_simple(t)` prepara un texto para compararlo
* `campo_real(capa, nombre)` recorre los campos de una capa y devuelve el nombre real del primero cuyo nombre o alias coincide con el buscado después de pasar los dos por `_simple(t)`
* `anillos(geom)` Saca las coordenadas de los bordes del polígono.  La función devuelve una lista con una entrada por parte, y cada entrada es una lista de anillos: primero el exterior y después los huecos, cada uno como arreglo de coordenadas X e Y.
* `dentro(partes, xy)`  Decide qué puntos de xy (los centros de las celdas) caen dentro del polígono. Para cada parte,  marca los puntos dentro del anillo exterior, y a esos se les quitan los que caen dentro de algún hueco. Un punto está dentro del polígono si está dentro de cualquiera de sus partes.
*
* `corona_de(z, xc, yc, partes)` Busca la celda de cota máxima dentro del polígono. Recibe el pedazo del DEM (z, con la primera fila al norte y NaN donde no hay dato) y las coordenadas de los centros de sus columnas (xc) y filas (yc). Devuelve las coordenadas y la cota de la corona, las celdas dentro del polígono, las celdas con dato y el número de celdas empatadas.



In [ ]:

def ruta(capa):
    return arcpy.Describe(capa).catalogPath

def _simple(t):
    t = unicodedata.normalize("NFKD", str(t)).encode("ascii", "ignore").decode()
    return t.strip().lower()

def campo_real(capa, nombre):
    """Busca el campo por nombre o alias, sin importar mayusculas ni tildes."""
    campos = arcpy.ListFields(capa)
    for f in campos:
        if _simple(f.name) == _simple(nombre) or _simple(f.aliasName) == _simple(nombre):
            return f.name
    raise ValueError("No encuentro el campo '{}' en {}.\n  Campos: {}".format(
        nombre, capa, ", ".join(f.name for f in campos)))

def anillos(geom):
    """Partes del poligono, cada una como [anillo exterior, huecos...]."""
    partes = []
    for parte in geom:
        lista, actual = [], []
        for p in parte:
            if p is None:                       # separador de anillos interiores
                if actual:
                    lista.append(np.array(actual))
                actual = []
            else:
                actual.append((p.X, p.Y))
        if actual:
            lista.append(np.array(actual))
        partes.append(lista)
    return partes

def dentro(partes, xy):
    """Celdas (centros xy) que caen dentro del poligono, respetando huecos."""
    res = np.zeros(len(xy), dtype=bool)
    for lista in partes:
        en_parte = Path(lista[0]).contains_points(xy)
        for hueco in lista[1:]:
            en_parte &= ~Path(hueco).contains_points(xy)
        res |= en_parte
    return res

def corona_de(z, xc, yc, partes):
    """Celda de cota maxima dentro del poligono.
    z: ventana del DEM (filas de norte a sur), con NaN en sin dato.
    xc, yc: coordenadas de los centros de columnas y filas de la ventana."""
    X, Y = np.meshgrid(xc, yc)
    xy = np.column_stack([X.ravel(), Y.ravel()])
    zz = z.ravel()
    ins = dentro(partes, xy)
    n_dentro = int(ins.sum())
    if n_dentro == 0:
        return None
    validas = ins & ~np.isnan(zz)
    n_validas = int(validas.sum())
    if n_validas == 0:
        return dict(n_celdas=n_dentro, n_validas=0)
    zmax = np.nanmax(zz[validas])
    emp = np.where(validas & (zz >= zmax - 1e-6))[0]
    if len(emp) > 1:                            # empate: la mas cercana al centro del grupo
        c = xy[emp].mean(axis=0)
        k = emp[np.argmin(((xy[emp] - c) ** 2).sum(axis=1))]
    else:
        k = emp[0]
    return dict(x=xy[k, 0], y=xy[k, 1], z=float(zmax), n_celdas=n_dentro,
                n_validas=n_validas, empates=len(emp))

# Corona de cada polígono

**¿Que hace?**

Recorre los polígonos del inventario. Para cada uno lee el pedazo del DEM que lo cubre, busca la corona  y aplica las exclusiones: sin celdas en el DEM, más del 10 % de las celdas sin dato, o menos de 10 celdas. Al final imprime cuántas coronas se calcularon y qué polígonos se excluyeron. 



In [ ]:


dem = arcpy.Raster(ruta(DEM))
sr  = dem.spatialReference
cs  = dem.meanCellWidth
X0, YTOP = dem.extent.XMin, dem.extent.YMax
NCOL, NFIL = dem.width, dem.height
sr_pol = arcpy.Describe(CAPA_POLIGONOS).spatialReference
coronas, sin_celdas, recortes, pocas, empates = {}, [], [], [], 0
if sr_pol.factoryCode != sr.factoryCode:
    print("ATENCION: los poligonos ({}) y el DEM ({}) tienen distinto sistema; "
          "se leen los poligonos en el del DEM.".format(sr_pol.factoryCode, sr.factoryCode))

f_id = campo_real(CAPA_POLIGONOS, CAMPO_ID_POLIGONOS)
with arcpy.da.SearchCursor(CAPA_POLIGONOS, [f_id, "SHAPE@"], spatial_reference=sr) as cur:
    for i, (pid, geom) in enumerate(cur, 1):
        e = geom.extent
        c0 = max(0, int(math.floor((e.XMin - X0) / cs)))
        c1 = min(NCOL, int(math.ceil((e.XMax - X0) / cs)))
        r0 = max(0, int(math.floor((YTOP - e.YMax) / cs)))
        r1 = min(NFIL, int(math.ceil((YTOP - e.YMin) / cs)))
        if c1 <= c0 or r1 <= r0:
            sin_celdas.append(pid)
            continue
        esquina = arcpy.Point(X0 + c0 * cs, YTOP - r1 * cs)
        z = arcpy.RasterToNumPyArray(dem, esquina, c1 - c0, r1 - r0,
                                     nodata_to_value=-99999).astype(float)
        z[z <= -99990] = np.nan
        xc = X0 + (np.arange(c0, c1) + 0.5) * cs
        yc = YTOP - (np.arange(r0, r1) + 0.5) * cs
        r = corona_de(z, xc, yc, anillos(geom))
        if r is None or r["n_validas"] == 0:
            sin_celdas.append(pid)
            continue
        falta = 1 - r["n_validas"] / r["n_celdas"]
        if falta > FRACCION_RECORTE:
            recortes.append((pid, round(100 * falta)))
            continue
        empates += r["empates"] > 1
        if r["n_celdas"] < MIN_CELDAS:
            pocas.append((pid, r["n_celdas"]))
            continue
        coronas[pid] = r
        if i % 100 == 0:
            print("  {} poligonos procesados".format(i))

print("\nCoronas calculadas:", len(coronas))
print("Sin celdas del DEM:", len(sin_celdas), "->", sin_celdas)
print("En parte fuera del DEM (excluidos, % de celdas sin dato):", recortes)
print("Con menos de {} celdas (excluidos):".format(MIN_CELDAS), pocas)
print("Poligonos con empate en la cota maxima (se eligio una celda):", empates)
manual = [i for i in EXCLUIR_MANUAL if i in coronas]
for i in manual:
    coronas.pop(i)
print("Excluidos a mano:", manual)

# Atributos y control de completitud

**¿Que hace?**

Reúne los atributos de cada evento: el área, del inventario, y el grupo, el tipo, la reactivación, H/L y la elongación, de la capa de la Fase 1. Con el área y el grupo calcula si el evento está sobre el umbral de completitud de la Fase 0, y comprueba que los conteos coincidan con los de esa fase. No escribe nada.

In [ ]:


def es_si(v):
    if v is None:
        return 0
    if isinstance(v, (int, float)):
        return int(v == 1)
    return int(_simple(v)[:1] in ("s", "y", "t", "1"))

# Area: de la geometria del inventario actualizado
area = {pid: a for pid, a in arcpy.da.SearchCursor(CAPA_POLIGONOS, [f_id, "SHAPE@AREA"])}

# Atributos: de la capa de metricas de la Fase 1
f_mid = campo_real(CAPA_METRICAS, CAMPO_ID_METRICAS)
f_gr, f_tp, f_re, f_hl, f_el = (campo_real(CAPA_METRICAS, c) for c in
                                (CAMPO_GRUPO, CAMPO_TIPO, CAMPO_REACTIV, CAMPO_HL, CAMPO_ELONG))
atrib = {}
with arcpy.da.SearchCursor(CAPA_METRICAS, [f_mid, f_gr, f_tp, f_re, f_hl, f_el]) as cur:
    for pid, gr, tp, re_, hl, el in cur:
        atrib[pid] = dict(grupo="Mapeo" if _simple(gr)[:1] == "m" else "Institucional",
                          tipo=tp, reactiv=es_si(re_), hl=hl, el=el)

faltan_attr = sorted(p for p in area if p not in atrib)
faltan_pol  = sorted(p for p in atrib if p not in area)
if faltan_attr:
    print("ATENCION: poligonos sin fila en la capa de metricas:", faltan_attr)
if faltan_pol:
    print("ATENCION: filas de la capa de metricas sin poligono:", faltan_pol)

conteo = {"Mapeo": [0, 0], "Institucional": [0, 0]}
for pid, a in area.items():
    if pid not in atrib:
        continue
    g = atrib[pid]["grupo"]
    atrib[pid]["area"] = a
    atrib[pid]["completo"] = int(a >= (UMBRAL_MAPEO if g == "Mapeo" else UMBRAL_INST))
    conteo[g][atrib[pid]["completo"]] += 1

print("Bajo el umbral, todos los poligonos -> Mapeo: {} de {} | Institucional: {} de {}".format(
    conteo["Mapeo"][0], sum(conteo["Mapeo"]), conteo["Institucional"][0], sum(conteo["Institucional"])))
print("Deben coincidir con la Fase 0: 50 de 302 y 35 de 151.")

sin_hl = sorted(p for p in coronas if p not in atrib or atrib[p]["hl"] is None)
print("Coronas sin H/L en la capa de metricas:", sin_hl)


# Capa `coronas_revision` y comparación con la anterior


**¿Qué hace?**

Escribe la capa de puntos `coronas_revision`, con una corona por evento y sus atributos. Si ya existía una versión, la renombra en lugar de borrarla y compara la posición de cada corona con la de esa versión.



In [ ]:
destino = os.path.join(GDB_SALIDA, NOMBRE_CAPA)
anteriores = {}
if arcpy.Exists(destino):
    try:
        f_ant = campo_real(destino, "ID_POL")
        anteriores = {pid: xy for pid, xy in arcpy.da.SearchCursor(destino, [f_ant, "SHAPE@XY"])}
    except ValueError:
        pass
    k = 1
    while arcpy.Exists(destino + "_v{}".format(k)):
        k += 1
    arcpy.management.Rename(destino, destino + "_v{}".format(k))
    print("La capa anterior quedo como {}_v{}".format(NOMBRE_CAPA, k))

arcpy.management.CreateFeatureclass(GDB_SALIDA, NOMBRE_CAPA, "POINT", spatial_reference=sr)
tipo_id = {"String": "TEXT", "Double": "DOUBLE", "Single": "DOUBLE"}.get(
    arcpy.ListFields(CAPA_POLIGONOS, f_id)[0].type, "LONG")      # mismo tipo que el ID original
campos = [("ID_POL", tipo_id), ("grupo", "TEXT"), ("HL", "DOUBLE"), ("elongacion", "DOUBLE"),
          ("logArea", "DOUBLE"), ("area_m2", "DOUBLE"), ("tipo_mov", "TEXT"), ("reactiv", "SHORT"),
          ("completo", "SHORT"), ("z_corona", "DOUBLE"), ("n_celdas", "LONG"),
          ("POINT_X", "DOUBLE"), ("POINT_Y", "DOUBLE")]
for nombre, tipo in campos:
    arcpy.management.AddField(destino, nombre, tipo, field_length=40 if tipo == "TEXT" else None)

with arcpy.da.InsertCursor(destino, ["SHAPE@XY"] + [c[0] for c in campos]) as ins:
    for pid in sorted(coronas):
        if pid not in atrib:
            continue
        r, a = coronas[pid], atrib[pid]
        hl, el = a["hl"], a["el"]
        ins.insertRow([(r["x"], r["y"]), pid, a["grupo"], hl, el, math.log10(a["area"]), a["area"],
                       a["tipo"], a["reactiv"], a["completo"], r["z"], r["n_celdas"], r["x"], r["y"]])
print("Escrita {} con {} coronas".format(NOMBRE_CAPA, sum(1 for p in coronas if p in atrib)))

if anteriores:
    comunes = [p for p in coronas if p in anteriores]
    dist = {p: math.hypot(coronas[p]["x"] - anteriores[p][0], coronas[p]["y"] - anteriores[p][1])
            for p in comunes}
    movidas = sorted(p for p, d in dist.items() if d > 2)
    print("\nComparacion con la capa anterior:")
    print("  Coronas que se movieron mas de 2 m: {} de {}".format(len(movidas), len(comunes)))
    if movidas:
        print("  Desplazamiento maximo: {:.0f} m".format(max(dist[p] for p in movidas)))
        print("  IDs:", movidas)
    print("  Coronas nuevas:", sorted(set(coronas) - set(anteriores)))
    print("  Coronas que ya no estan:", sorted(set(anteriores) - set(coronas)))

# Pendiente en la corona

**¿Que hace?**

Agrega a `coronas_revision` el campo `pendiente`, con la pendiente del terreno en la corona, en grados, tomada de un ráster de pendiente con celdas de 10 m. Imprime la mediana, el percentil 1 y cuántas coronas tienen menos de 10°. 


In [ ]:

arcpy.CheckOutExtension("Spatial")
if RASTER_PENDIENTE is None:
    dem10 = arcpy.sa.Aggregate(dem, int(round(10 / cs)), "MEAN", "EXPAND", "DATA")
    pend = arcpy.sa.Slope(dem10, "DEGREE")
    RASTER_PENDIENTE = os.path.join(GDB_SALIDA, "pendiente_10m_calc")
    pend.save(RASTER_PENDIENTE)
arcpy.sa.ExtractMultiValuesToPoints(destino, [[RASTER_PENDIENTE, "pendiente"]], "NONE")

p = np.array([v for (v,) in arcpy.da.SearchCursor(destino, ["pendiente"]) if v is not None])
print("Pendiente en las coronas: mediana {:.1f} | percentil 1 = {:.1f} grados | < 10 grados: {}".format(
    np.median(p), np.percentile(p, 1), int((p < 10).sum())))
print("Antes: mediana 34,5 y percentil 1 = 4,7 grados (define la ventana de sensibilidad).")

#  Exportación de `coronas.shp` para R

**¿Que hace?**

Copia `coronas_revision` como shapefile, `coronas.shp`, en la carpeta de los scripts de R, e imprime cuántas coronas hay por grupo, por tipo de movimiento y cuántas son reactivaciones.


In [41]:

shp = os.path.join(CARPETA_R, "coronas.shp")
respaldo = os.path.join(CARPETA_R, "coronas_anterior.shp")
if arcpy.Exists(shp) and not arcpy.Exists(respaldo):
    arcpy.management.CopyFeatures(shp, respaldo)
    print("La version anterior quedo como coronas_anterior.shp")
arcpy.management.CopyFeatures(destino, shp)

from collections import Counter
filas = list(arcpy.da.SearchCursor(destino, ["grupo", "tipo_mov", "reactiv"]))
print("coronas.shp:", len(filas), "coronas")
print("  Por grupo:", dict(Counter(f[0] for f in filas)))
print("  Por tipo:", dict(Counter(f[1] for f in filas)))
print("  Reactivaciones:", sum(f[2] for f in filas))

Geodatabase de salida: C:\Users\villa\Documents\ArcGIS\Projects\GDB_proyecto de grado\GDB_proyecto de grado.gdb
  100 poligonos procesados
  200 poligonos procesados
  300 poligonos procesados
  400 poligonos procesados

Coronas calculadas: 442
Sin celdas del DEM: 5 -> [222, 360, 361, 362, 363]
En parte fuera del DEM (excluidos, % de celdas sin dato): [(364, 60)]
Con menos de 10 celdas (excluidos): [(56, 7), (75, 9), (93, 4), (109, 9), (440, 8)]
Poligonos con empate en la cota maxima (se eligio una celda): 5
Excluidos a mano: []
Bajo el umbral, todos los poligonos -> Mapeo: 50 de 302 | Institucional: 35 de 151
Deben coincidir con la Fase 0: 50 de 302 y 35 de 151.
Coronas sin H/L en la capa de metricas: []
La capa anterior quedo como coronas_revision_v3
Escrita coronas_revision con 442 coronas

Comparacion con la capa anterior:
  Coronas que se movieron mas de 2 m: 0 de 442
  Coronas nuevas: []
  Coronas que ya no estan: [56, 75, 93, 109, 440]
Pendiente en las coronas: mediana 34.5 | pe